In [1]:
# Import packages and modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

In [2]:
# Check the version of XGBoost
print("Found XGBoost v" + xgboost.__version__)

Found XGBoost v2.1.4


In [3]:
datasetPath = "dataset/task2/"

In [4]:
# Parameters
datasetPath = "dataset/577p/firstorder/kernel5/"


In [5]:
# train/validation/test = 70/10/20
train_data = pd.read_csv(f"{datasetPath}dataset.train.csv")
# validation_data = pd.read_csv(f"{datasetPath}dataset.validation.csv")
# test_data = pd.read_csv(f"{datasetPath}dataset.test.csv")

## 5-fold cross-validation at patient level

Cross-validation is performed only on the development set (**Train + Validation**). The independent **Test** set is not used during cross-validation or model selection. All voxels from the same patient are kept in the same fold by `StratifiedGroupKFold`.


In [6]:
# 5-fold patient-level cross-validation (Train + Validation only)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix
)

# Detect the patient identifier column. Add another candidate here if your CSV uses a different name.
group_candidates = [
    "patient_id", "patient", "case_id", "case",
    "subject_id", "subject", "BraTS21ID", "brats_id"
]
group_col = next((c for c in group_candidates if c in train_data.columns), None)
if group_col is None:
    raise ValueError(
        "Patient ID column was not found. "
        "Set group_col manually to the column identifying each patient, "
        "for example: group_col = 'patient_id'."
    )

print(f"Patient/group column: {group_col}")

# IMPORTANT: keep the independent test set completely outside CV.
dev_data = pd.concat([train_data], ignore_index=True)

# Exclude labels, weights, and patient/case metadata from model features.
metadata_cols = set(group_candidates + ["label", "sample_weight"])
cv_feature_cols = [c for c in dev_data.columns if c not in metadata_cols]

X_dev = dev_data[cv_feature_cols]
y_dev = dev_data["label"].astype(int)
groups_dev = dev_data[group_col]

# Sanity check: no patient in development data may occur in the independent test set.
# if group_col in test_data.columns:
#     overlap = set(groups_dev.unique()) & set(test_data[group_col].unique())
#     assert len(overlap) == 0, (
#         f"Data leakage: {len(overlap)} patient(s) occur in both development and test sets: "
#         f"{list(overlap)[:10]}"
#     )

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_dev, y_dev, groups=groups_dev), start=1
):
    X_fold_train = X_dev.iloc[train_idx]
    y_fold_train = y_dev.iloc[train_idx]
    X_fold_val = X_dev.iloc[val_idx]
    y_fold_val = y_dev.iloc[val_idx]

    train_groups = set(groups_dev.iloc[train_idx])
    val_groups = set(groups_dev.iloc[val_idx])
    assert train_groups.isdisjoint(val_groups), f"Patient leakage in fold {fold}"

    # Compute class weights from this fold's training partition only.
    fold_classes = np.unique(y_fold_train)
    fold_weights = compute_class_weight(
        class_weight="balanced",
        classes=fold_classes,
        y=y_fold_train
    )
    fold_class_weight = dict(zip(fold_classes, fold_weights))
    fold_sample_weight = y_fold_train.map(fold_class_weight).to_numpy()

    cv_model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric=["logloss", "auc"],
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    cv_model.fit(
        X_fold_train,
        y_fold_train,
        sample_weight=fold_sample_weight,
        verbose=False
    )

    fold_prob = cv_model.predict_proba(X_fold_val)[:, 1]
    fold_pred = (fold_prob >= 0.5).astype(int)

    cm = confusion_matrix(y_fold_val, fold_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    result = {
        "fold": fold,
        "train_patients": len(train_groups),
        "val_patients": len(val_groups),
        "train_voxels": len(train_idx),
        "val_voxels": len(val_idx),
        "accuracy": accuracy_score(y_fold_val, fold_pred),
        "precision": precision_score(y_fold_val, fold_pred, zero_division=0),
        "recall": recall_score(y_fold_val, fold_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_fold_val, fold_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_fold_val, fold_prob),
        "pr_auc": average_precision_score(y_fold_val, fold_prob),
        "mcc": matthews_corrcoef(y_fold_val, fold_pred),
    }
    fold_results.append(result)

    print(
        f"Fold {fold}: "
        f"patients(train/val)={len(train_groups)}/{len(val_groups)}, "
        f"AUC={result['roc_auc']:.4f}, F1={result['f1']:.4f}, "
        f"Recall={result['recall']:.4f}"
    )

cv_results = pd.DataFrame(fold_results)
display(cv_results)

metric_cols = [
    "accuracy", "precision", "recall", "specificity",
    "f1", "roc_auc", "pr_auc", "mcc"
]
cv_summary = pd.DataFrame({
    "mean": cv_results[metric_cols].mean(),
    "std": cv_results[metric_cols].std(ddof=1),
})

print("\n5-fold patient-level CV summary (mean ± SD):")
for metric in metric_cols:
    print(
        f"{metric:12s}: "
        f"{cv_summary.loc[metric, 'mean']:.4f} ± "
        f"{cv_summary.loc[metric, 'std']:.4f}"
    )

display(cv_summary)

Patient/group column: patient_id


Fold 1: patients(train/val)=461/116, AUC=0.9276, F1=0.4385, Recall=0.8215


Fold 2: patients(train/val)=462/115, AUC=0.8959, F1=0.3696, Recall=0.7939


Fold 3: patients(train/val)=463/114, AUC=0.9197, F1=0.4483, Recall=0.8168


Fold 4: patients(train/val)=460/117, AUC=0.9064, F1=0.3970, Recall=0.7914


Fold 5: patients(train/val)=462/115, AUC=0.9185, F1=0.4346, Recall=0.8125


,fold,train_patients,val_patients,train_voxels,val_voxels,accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,mcc
0,1,461,116,521577,131820,0.872440,0.299034,0.821549,0.875724,0.438471,0.927557,0.610041,0.446608
1,2,462,115,523132,130265,0.830223,0.240832,0.793876,0.832654,0.369555,0.895875,0.525098,0.375081
2,3,463,114,523605,129792,0.863412,0.308943,0.816759,0.866813,0.448310,0.919677,0.615521,0.448119
3,4,460,117,521845,131552,0.846775,0.264944,0.791388,0.850546,0.396984,0.906427,0.561179,0.399419
4,5,462,115,523429,129968,0.866036,0.296613,0.812508,0.869657,0.434579,0.918502,0.620115,0.438786



5-fold patient-level CV summary (mean ± SD):
accuracy    : 0.8558 ± 0.0171
precision   : 0.2821 ± 0.0284
recall      : 0.8072 ± 0.0137
specificity : 0.8591 ± 0.0175
f1          : 0.4176 ± 0.0332
roc_auc     : 0.9136 ± 0.0125
pr_auc      : 0.5864 ± 0.0416
mcc         : 0.4216 ± 0.0327


,mean,std
accuracy,0.855777,0.017141
precision,0.282073,0.028352
recall,0.807216,0.013720
specificity,0.859079,0.017463
f1,0.417580,0.033167
roc_auc,0.913608,0.012464
pr_auc,0.586391,0.041646
mcc,0.421602,0.032711
